In [ ]:
from typing import List

from pydantic import BaseModel, Field, field_validator

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()
# Retrieve the Gemini API key from Colab secrets
groq_api_key = os.getenv('GROQ_API_KEY')

# Initialize the client with the retrieved API key
# groq = client.Client(api_key=groq_api_key)


# --------------------------------------------------
# 1. INPUT SIMULATION
# --------------------------------------------------

resumes = [
    """
    Rahul Sharma is a backend developer with 8 years of experience.
    He has worked extensively with Ruby on Rails, PostgreSQL, Redis,
    AWS and Docker.
    """,

    """
    Priya Patel is a software engineer with 6 years of experience.
    Her skills include Python, Django, FastAPI, PostgreSQL and AWS.
    """,

    """
    Amit Kumar is a senior Java developer with 10 years of experience.
    He specializes in Java, Spring Boot, Kafka, Kubernetes and AWS.
    """,

    """
    Sneha Desai has 5 years of experience as a frontend developer.
    She works with JavaScript, React, TypeScript, HTML and CSS.
    """,

    """
    Vikram Singh is a DevOps engineer with 7 years of experience.
    His main skills are Docker, Kubernetes, Jenkins, Terraform and AWS.
    """,

    """
    Neha Joshi is a data engineer with 9 years of experience.
    She has expertise in Python, SQL, Spark, Airflow and AWS.
    """,

    # INVALID RESUME
    """
    Looking for opportunities in the software industry.
    I have worked on different technologies and projects.
    I am interested in learning new skills.
    """
]


# --------------------------------------------------
# 2. PYDANTIC MODEL
# --------------------------------------------------

class Resume(BaseModel):

    name: str = Field(description="Candidate's full name")
    skills: List[str] = Field(description="Candidate's skills")
    experience: int = Field(
        description="Total years of professional experience"
    )

    @field_validator("name")
    @classmethod
    def validate_name(cls, value):
        if not value or value.lower() in ["unknown", "n/a", "not provided"]:
            raise ValueError("Candidate name is missing")
        return value

    @field_validator("experience")
    @classmethod
    def validate_experience(cls, value):
        if value <= 0:
            raise ValueError("Experience must be greater than 0")
        return value


# --------------------------------------------------
# 3. PARSER
# --------------------------------------------------

parser = PydanticOutputParser(
    pydantic_object=Resume
)


# --------------------------------------------------
# 4. PROMPT
# --------------------------------------------------

prompt = PromptTemplate(
    template="""
Extract structured information from this resume.

Resume:
{resume}

Return the candidate's:

- Name
- Skills
- Total years of experience

{format_instructions}
""",
    input_variables=["resume"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)


# --------------------------------------------------
# 5. LLM
# --------------------------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=groq_api_key
)


# --------------------------------------------------
# 6. CHAIN
# --------------------------------------------------

chain = prompt | llm | parser


# --------------------------------------------------
# 7. BATCH PROCESSING
# --------------------------------------------------

inputs = [
    {"resume": resume}
    for resume in resumes
]

print("Processing resumes...\n")

# results = chain.batch(inputs)
# response = llm.invoke("Hello")

# print(results)
results = []

for index, input_data in enumerate(inputs, start=1):

    try:
        result = chain.invoke(input_data)

        results.append({
            "index": index,
            "result": result,
            "status": "VALIDATED",
            "error": None
        })

    except Exception as e:

        results.append({
            "index": index,
            "result": None,
            "status": "INVALIDATED",
            "error": str(e)
        })


# --------------------------------------------------
# 8. REAL-TIME DISPLAY
# --------------------------------------------------

for item in results:

    print(f"Resume {item['index']}")
    print("-" * 40)

    if item["status"] == "VALIDATED":

        result = item["result"]

        print(f"Name       : {result.name}")
        print(f"Skills     : {', '.join(result.skills)}")
        print(f"Experience : {result.experience} years")
        print("Status     : VALIDATED")

    else:

        print("Status     : INVALIDATED")
        print("Reason     : Resume failed Pydantic validation")

    print()

Processing resumes...

Resume 1
----------------------------------------
Name       : Rahul Sharma
Skills     : Ruby on Rails, PostgreSQL, Redis, AWS, Docker
Experience : 8 years
Status     : VALIDATED

Resume 2
----------------------------------------
Name       : Priya Patel
Skills     : Python, Django, FastAPI, PostgreSQL, AWS
Experience : 6 years
Status     : VALIDATED

Resume 3
----------------------------------------
Name       : Amit Kumar
Skills     : Java, Spring Boot, Kafka, Kubernetes, AWS
Experience : 10 years
Status     : VALIDATED

Resume 4
----------------------------------------
Name       : Sneha Desai
Skills     : JavaScript, React, TypeScript, HTML, CSS
Experience : 5 years
Status     : VALIDATED

Resume 5
----------------------------------------
Name       : Vikram Singh
Skills     : Docker, Kubernetes, Jenkins, Terraform, AWS
Experience : 7 years
Status     : VALIDATED

Resume 6
----------------------------------------
Name       : Neha Joshi
Skills     : Python, S

In [3]:
from groq import Groq

client = Groq(api_key=groq_api_key)

for model in client.models.list().data:
    print(model.id)



meta-llama/llama-prompt-guard-2-86m
allam-2-7b
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
openai/gpt-oss-120b
whisper-large-v3
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="meta-llama/llama-prompt-guard-2-86m",
    temperature=0
)

response = llm.invoke("Hello")

print(response.content)

test_input = {
    "resume": resumes[0]
}


result = chain.invoke(test_input)

print(result)

OutputParserException: Failed to parse Resume from completion 0.0024326585698872805. Got: 1 validation error for Resume
  Input should be a valid dictionary or instance of Resume [type=model_type, input_value=0.0024326585698872805, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()
# Retrieve the Gemini API key from Colab secrets
groq_api_key = os.getenv('GROQ_API_KEY')
# print("API key exists:", groq_api_key is not None)
# print("API key length:", len(groq_api_key) if groq_api_key else 0)

llm = ChatGroq(
    model="meta-llama/llama-prompt-guard-2-86m",
    temperature=0,
    api_key=groq_api_key
)

response = llm.invoke(
    """
    Extract the following resume information and return ONLY JSON.

    Resume:
    Rahul Sharma is a backend developer with 8 years of experience.
    He works with Ruby on Rails, PostgreSQL, Redis and AWS.

    JSON format:
    {
      "name": "string",
      "skills": ["string"],
      "experience": 0
    }
    """
)

print(type(response))
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
0.0004330166266299784
